# 固定 t 的 loss 方差分解：大 t 抖动是测量噪声还是任务本质不稳定？

**研究问题**：Dt 谱在大 t 处抖动剧烈——是"粒度过大、任务本质不稳定"（假说），还是单纯测量噪声（平滑即可）？

**关键陷阱**：loss 的绝对量级随 t 单调增长（可差一个数量级以上），"原始方差随 t 增大"数学上必然成立、什么都证明不了。判据必须用相对量：
- CV = std/|mean|（变异系数，消除量纲）——L₀、Lz 用；
- SNR = mean(D)/std(D)——D 的均值可跨 0、CV 会发散，D 以 SNR 为主判据（这正是控制器实际吃到的信噪比）。

**三臂方差分解**（模型冻结：eval + EMA shadow，消除训练动态混淆）：

| 臂 | 固定 | 随机 | 回答什么 |
|---|---|---|---|
| A | 同一批 x₀、同一 t | 仅噪声 ε | 噪声采样本身贡献多少方差 |
| B | 同一 ε、同一 t | 仅数据 x₀ | 数据多样性贡献多少方差 |
| C | t | x₀ 和 ε 都随机 | 训练时实际面对的总方差（主结果） |

**配对采样**：两分支 L(xt,t,0) 与 L(xt,t,z_sem) 共享同一个 x_t（同 x₀、同 ε），
Var(D) = Var(L₀)+Var(Lz)−2Cov 中的协方差被保留——这是 D 方差远小于独立假设的主因。

**判定标准**：
- CV 随 t 基本平 → 抖动是量纲效应，"粒度过大"假说不成立，控制器不用改，平滑即可；
- CV 上升但 D 的 SNR 不崩 → 测量端加 EMA/多采样就能解决；
- CV 上升且大 t 处 SNR 崩溃（|SNR|<1）→ 假说成立，LCB 风险项有依据，SNR(t) 曲线直接给出 κ·σ_t 的标定。

In [ ]:
import os, sys
import torch

# ---- 先在 notebook 所在目录（visual/）下把两个路径都算成绝对路径 ----
VISUAL = os.path.abspath('.')                                  # 内核启动时 cwd 就是 visual/
MMSEG  = os.path.abspath('../tasks/Segmentation/mmsegmentation')

os.chdir(MMSEG)
for p in [MMSEG, os.path.join(MMSEG, 'eden'), VISUAL]:
    if p not in sys.path:
        sys.path.insert(0, p)

# eden 包在服务器上部署名为 mydiffae：本地别名兼容
import eden, eden.diffae_mixin
sys.modules.setdefault('mydiffae', eden)
sys.modules.setdefault('mydiffae.diffae_mixin', eden.diffae_mixin)

# 注册 adapter（FCNWithDiffAEPlusPCGrad）；已注册过则跳过
try:
    import mmseg.models.segmentors.eden_adapter_fcn  # noqa
except KeyError:
    pass

import probe_core as pc

In [ ]:
# ---- 配置：改成你的 run 目录 ----
# CKPTS 取早/中/晚三个（方差结构随训练演化）；SEED 固定 => 噪声与分组跨 ckpt 完全
# 相同（CRN），三条曲线严格可比。NUM_IMAGES=64：臂 B/C 的数据池；臂 A 固定前 16 张。
# 每 t 样本量：A = 32ε×16图，B = 8ε×64图，C = 8轮×64图 = 512（≥128 的方差估计需求）。
import glob, re
import numpy as np
RUN_DIR = 'work_dirs/eden/fcn_diffae'
CFG = sorted(glob.glob(os.path.join(RUN_DIR, '*.py')))[0]   # run 目录里 dump 的 config
ALL_CKPTS = sorted(glob.glob(f'{RUN_DIR}/iter_*.pth'),
                   key=lambda p: int(re.findall(r'\d+', os.path.basename(p))[-1]))
pick = np.unique(np.linspace(0, len(ALL_CKPTS) - 1, 3).round().astype(int))  # 早/中/晚
CKPTS = [ALL_CKPTS[i] for i in pick]
OUT = 'work_dirs/offline_probe/loss_var'
NUM_IMAGES = 64
N_T = 32            # t 网格等距取点数（20~50）
N_EPS_A = 32        # 臂 A 每图噪声重复数
N_NOISE_B = 8       # 臂 B 固定 ε 个数（每个扫全部图）
N_ROUNDS_C = 8      # 臂 C 轮数（每轮扫全部图，每块新噪声）
SEED = 0
os.makedirs(OUT, exist_ok=True)
print('ckpts:', [os.path.basename(p) for p in CKPTS])

In [ ]:
# 数据只取一次，所有 ckpt 共用（同图同分布，跨 ckpt 可比）
# 注意：batch['cond'] 是随机初始化 backbone 算的，本实验不用它——
# measure_loss_variance 内部对每个 ckpt 现算 cond（训练同分布的 z_sem）。
model, cfg = pc.build_model(CFG)   # 先建一次只为取数
batch = pc.build_val_batch(model, cfg, NUM_IMAGES)
del model; torch.cuda.empty_cache()

In [ ]:
# 主循环：每个 ckpt 冻结测量 -> 六联图 + 原始数据 npz + 判定读数
results, labels, verdicts = [], [], []
for ckpt in CKPTS:
    model, _ = pc.build_model(CFG, ckpt)
    tag = os.path.splitext(os.path.basename(ckpt))[0]
    res = pc.measure_loss_variance(
        model, batch, n_t=N_T, n_eps_a=N_EPS_A, n_noise_b=N_NOISE_B,
        n_rounds_c=N_ROUNDS_C, use_ema=True, seed=SEED)
    fig = pc.plot_loss_variance(
        res, title=f'{tag} | loss variance decomposition',
        save_path=os.path.join(OUT, f'{tag}_lossvar.png'))
    # 原始数组全部落盘（供复核/再分析）
    flat = {}
    def _flat(d, p=''):
        for k, v in d.items():
            if isinstance(v, dict):
                _flat(v, p + k + '__')
            else:
                flat[p + k] = v
    _flat(res)
    np.savez(os.path.join(OUT, f'{tag}_lossvar.npz'), **flat)
    v = pc.lossvar_verdict(res)
    results.append(res); labels.append(tag); verdicts.append(v)
    print(f"{tag}: Spearman(CV_D,t)={v['rho_cv_d'][0]:.3f}(p={v['rho_cv_d'][1]:.3f}) | "
          f"Spearman(CV_Lz,t)={v['rho_cv_lz'][0]:.3f} | "
          f"min|SNR(D)|@high-t={v['min_snr_high_t']:.3f} | "
          f"corr(L0,Lz)={v['mean_corr_l0_lz']:.3f}")
    del model; torch.cuda.empty_cache()
fig  # 显示最后一张

In [ ]:
# 跨 ckpt 汇总：CV(D) / |SNR(D)| 演化 + 判定表
if len(results) > 1:
    fig_evo = pc.plot_loss_variance_evolution(
        results, labels, save_path=os.path.join(OUT, 'evolution.png'))
    display(fig_evo)

header = f"{'ckpt':<14} {'rho(CV_D,t)':<18} {'rho(CV_Lz,t)':<13} {'rho(CV_L0,t)':<13} {'min|SNR| hi-t':<14} {'corr(L0,Lz)':<11}"
print(header); print('-' * len(header))
for lab, v in zip(labels, verdicts):
    print(f"{lab:<14} {v['rho_cv_d'][0]:>6.3f} (p={v['rho_cv_d'][1]:.3f})   "
          f"{v['rho_cv_lz'][0]:>10.3f}   {v['rho_cv_l0'][0]:>10.3f}   "
          f"{v['min_snr_high_t']:>11.3f}   {v['mean_corr_l0_lz']:>9.3f}")
print()
print('判定映射：')
print('  rho(CV,t)≈0                    -> 量纲效应，控制器不用改，平滑即可')
print('  rho(CV,t)>0 且 min|SNR|>=1     -> 测量端问题，加 EMA/多采样')
print('  rho(CV,t)>0 且 min|SNR|<1      -> 假说成立，SNR(t) 可标定 LCB 的 κ·σ_t')

In [ ]:
# 可选第四臂：固定 t 下 cos<g_task, g_diff> 的逐实现方差（与 meta-grad 红线抖动直接对应）
# 代价 = n_t×n_noise 次 backbone 反传，默认关闭；开启后只在中间 ckpt 上跑。
RUN_GRAD_COS = False
if RUN_GRAD_COS:
    ckpt = CKPTS[len(CKPTS) // 2]
    model, _ = pc.build_model(CFG, ckpt)
    tag = os.path.splitext(os.path.basename(ckpt))[0]
    gres = pc.measure_grad_cos_variance(model, batch, n_t=16, n_noise=16, n_images=8, seed=SEED)
    fig_g = pc.plot_grad_cos_variance(
        gres, title=f'{tag} | cos<g_task, g_diff(t)> per-realization',
        save_path=os.path.join(OUT, f'{tag}_gradcos_var.png'))
    np.savez(os.path.join(OUT, f'{tag}_gradcos.npz'), **gres)
    rho, p = pc.spearman(gres['t'], gres['cos_std'])
    print(f'{tag}: Spearman(std(cos), t) = {rho:.3f} (p={p:.3f})  # >0 显著 = 大 t 梯度方向本身不稳')
    display(fig_g)
    del model; torch.cuda.empty_cache()